# 03 Responses API Debug: gpt-5.3-codex

目标：用 TokenLab 的 OpenAI-compatible endpoint 调试 `gpt-5.3-codex` 的 **Responses API** 返回格式，并验证它不只是 coding 专用模型。

本 notebook 参考 `02_sandbox.ipynb` 的 SDK 初始化方式，但重点不是 sandbox 改代码，而是观察：

- `client.responses.create(...)` 非流式 Response object
- `client.responses.create(..., stream=True)` 流式事件格式
- `OpenAIResponsesModel` 在 Agents SDK 里的事件格式
- Responses API function tool call 的输出结构

> 不在 notebook 里硬编码 API key。运行前请在环境变量里设置：
> `OPENAI_API_KEY` 和 `OPENAI_BASE_URL=https://api.tokenlab.sh/v1`。


In [1]:
import asyncio
import json
import os
import time
from collections import Counter

from agents import Agent, OpenAIResponsesModel, Runner
from agents.run import RunConfig
from openai import AsyncOpenAI, OpenAI

MODEL = "gpt-5.3-codex"
BASE_URL = "https://api.tokenlab.sh/v1"
API_KEY = "sk-aeRemEo2sD0YgQWEFGjipWrzTp4LVFUVzHD8bD5fx5PoLMGF"

client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=90.0, max_retries=0)
async_client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=90.0, max_retries=0)

def dump(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    return json.loads(json.dumps(obj, default=str))

def pretty(obj, limit=6000):
    text = json.dumps(dump(obj), ensure_ascii=False, indent=2)
    print(text[:limit])
    if len(text) > limit:
        print(f"\n... truncated, total chars={len(text)}")

def ms(start):
    return int((time.perf_counter() - start) * 1000)

print("base_url=", BASE_URL)
print("model=", MODEL)


base_url= https://api.tokenlab.sh/v1
model= gpt-5.3-codex


## 1. Raw Responses API：非流式格式

观察重点：

- 顶层对象是 `response`
- 文本在 `output[0].content[0].text`
- `model` 会被网关展开成具体版本，例如 `gpt-5.3-codex-2026-02-24`
- usage 是 Responses API 格式：`input_tokens` / `output_tokens` / `output_tokens_details.reasoning_tokens`


In [2]:
start = time.perf_counter()
resp = client.responses.create(
    model=MODEL,
    input="用中文一句话说明你不只是编码模型。",
    max_output_tokens=80,
)
print("latency_ms=", ms(start))
pretty(resp)

# 便捷提取最终文本
output_text = ""
for item in resp.output:
    if getattr(item, "type", None) == "message":
        for part in getattr(item, "content", []) or []:
            if getattr(part, "type", None) == "output_text":
                output_text += getattr(part, "text", "")
print("\nextracted_text=", output_text)


latency_ms= 2584
{
  "id": "resp_024bf3475756827b006a434db846d081909af59d3b364edf2d",
  "created_at": 1782795704.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.3-codex-2026-02-24",
  "object": "response",
  "output": [
    {
      "id": "msg_024bf3475756827b006a434db8b8048190a0ed6c0bde4d8442",
      "content": [
        {
          "annotations": [],
          "text": "我不只是编码模型，我还能进行多领域的理解、推理、写作与交流来帮助你解决各种问题。",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 0.98,
  "background": false,
  "completed_at": null,
  "conversation": null,
  "max_output_tokens": 80,
  "max_tool_calls": null,
  "moderation": null,
  "previous_response_id": null,
  "prompt": null,
  "prompt_cache_key": null,

## 2. Raw Responses API：流式事件格式

观察重点：

- 文本 delta 事件是 `response.output_text.delta`
- delta 字段在 `event.delta`
- 完整文本会在 `response.output_text.done` 和最终 `response.completed` 里再次出现
- 该模型在 Responses API 下是 token / 小片段级流式


In [3]:
start = time.perf_counter()
stream = client.responses.create(
    model=MODEL,
    input="用中文一句话说明你不只是编码模型。",
    max_output_tokens=100,
    stream=True,
)

event_counter = Counter()
deltas = []
first_delta_ms = None

for i, event in enumerate(stream, 1):
    data = dump(event)
    event_type = data.get("type", "unknown")
    event_counter[event_type] += 1

    if event_type == "response.output_text.delta":
        if first_delta_ms is None:
            first_delta_ms = ms(start)
        delta = data.get("delta", "")
        deltas.append(delta)
        print(repr(delta), end=" ")

print("\n\nlatency_ms=", ms(start))
print("first_delta_ms=", first_delta_ms)
print("event_counter=", dict(event_counter))
print("text=", "".join(deltas))


'我' '不' '只是' '编码' '模型' '，我' '还能' '进行' '多' '领域' '的' '理解' '、' '推' '理' '、' '写' '作' '与' '问题' '解决' '。' 

latency_ms= 2277
first_delta_ms= 2245
event_counter= {'response.created': 1, 'response.in_progress': 1, 'response.output_item.added': 1, 'response.content_part.added': 1, 'response.output_text.delta': 22, 'response.output_text.done': 1, 'response.content_part.done': 1, 'response.output_item.done': 1, 'response.completed': 1}
text= 我不只是编码模型，我还能进行多领域的理解、推理、写作与问题解决。


## 3. 打印前若干个 Responses stream event 原始结构

用来对齐 adapter 时非常有用。重点看：

- `response.output_item.added`
- `response.content_part.added`
- `response.output_text.delta`
- `response.output_text.done`
- `response.completed`


In [4]:
stream = client.responses.create(
    model=MODEL,
    input="用中文回答：Responses API 的流式文本事件叫什么？",
    max_output_tokens=80,
    stream=True,
)

for i, event in enumerate(stream, 1):
    data = dump(event)
    print(f"\n--- EVENT {i}: {data.get('type')} ---")
    print(json.dumps(data, ensure_ascii=False, indent=2)[:1800])
    if i >= 12:
        break



--- EVENT 1: response.created ---
{
  "response": {
    "id": "resp_0f5af936927c7e58006a434dc732b88197b3165092263e364d",
    "created_at": 1782795719.0,
    "error": null,
    "incomplete_details": null,
    "instructions": null,
    "metadata": {},
    "model": "gpt-5.3-codex-2026-02-24",
    "object": "response",
    "output": [],
    "parallel_tool_calls": true,
    "temperature": 1.0,
    "tool_choice": "auto",
    "tools": [],
    "top_p": 0.98,
    "background": false,
    "completed_at": null,
    "conversation": null,
    "max_output_tokens": 80,
    "max_tool_calls": null,
    "moderation": null,
    "previous_response_id": null,
    "prompt": null,
    "prompt_cache_key": null,
    "prompt_cache_retention": "in_memory",
    "reasoning": {
      "context": null,
      "effort": "none",
      "generate_summary": null,
      "summary": null
    },
    "safety_identifier": null,
    "service_tier": "auto",
    "status": "in_progress",
    "text": {
      "format": {
        "typ

## 4. Responses API function tool call 格式

Responses API 的 function tool call 和 Chat Completions 的结构不同：

- 输出 item 类型是 `function_call`
- 字段通常是 `call_id` / `name` / `arguments`
- 不在 `choices[0].message.tool_calls` 里


In [5]:
tool_resp = client.responses.create(
    model=MODEL,
    input="请调用 multiply 工具计算 7*6，不要直接回答。",
    max_output_tokens=120,
    tools=[{
        "type": "function",
        "name": "multiply",
        "description": "Multiply two integers.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"},
            },
            "required": ["a", "b"],
            "additionalProperties": False,
        },
    }],
)
pretty(tool_resp)

print("\noutput items:")
for item in tool_resp.output:
    print(dump(item))


{
  "id": "resp_015528f0eaf7337e006a434dccb43c819597e241d05446caa9",
  "created_at": 1782795724.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5.3-codex-2026-02-24",
  "object": "response",
  "output": [
    {
      "arguments": "{\"a\":7,\"b\":6}",
      "call_id": "call_sXediFvNLI9lMX6VfOKZ07Du",
      "name": "multiply",
      "type": "function_call",
      "id": "fc_015528f0eaf7337e006a434dcd91f08195939936c8a555c2f1",
      "namespace": null,
      "status": "completed"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "multiply",
      "parameters": {
        "additionalProperties": false,
        "type": "object",
        "properties": {
          "a": {
            "type": "integer"
          },
          "b": {
            "type": "integer"
          }
        },
        "required": [
          "a",
          "b"
        ]
      },
      "strict":

## 5. Agents SDK：OpenAIResponsesModel 非流式

这里验证 `gpt-5.3-codex` 可以走 Agents SDK 的 Responses model 适配。


In [6]:
responses_model = OpenAIResponsesModel(
    model=MODEL,
    openai_client=async_client,
)

agent = Agent(
    name="gpt53-codex-responses-debug",
    model=responses_model,
    instructions="你是一个通用中文助手，不只是代码助手。回答要简洁。",
)

start = time.perf_counter()
result = await Runner.run(
    agent,
    "用中文回答：你只能写代码吗？",
    max_turns=3,
    run_config=RunConfig(tracing_disabled=True),
)
print("latency_ms=", ms(start))
print("final_output=", result.final_output)
print("new_items=", [type(item).__name__ for item in result.new_items])
for item in result.new_items:
    print("\n---", type(item).__name__, "---")
    print(str(item)[:2000])


latency_ms= 2813
final_output= 不，我不只能写代码。  
我也可以用中文帮你做很多事，比如：

- 解答知识问题
- 写作与润色（邮件、总结、文案）
- 翻译与改写
- 学习辅导与题目讲解
- 制定计划（学习、工作、旅行等）
- 头脑风暴和信息整理

你可以直接告诉我你想做什么。
new_items= ['MessageOutputItem']

--- MessageOutputItem ---
MessageOutputItem(agent=Agent(name='gpt53-codex-responses-debug', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='你是一个通用中文助手，不只是代码助手。回答要简洁。', prompt=None, handoffs=[], model=<agents.models.openai_responses.OpenAIResponsesModel object at 0x7fa5694e5e90>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], output_guardrails=[], output_typ

## 6. Agents SDK：OpenAIResponsesModel 流式事件

观察 Agents SDK 包装后的事件。adapter 层通常要处理：

- `raw_response_event` + `data.type == response.output_text.delta`
- `run_item_stream_event` / `message_output_created`
- tool call 时还会有 function/tool item


In [7]:
responses_model = OpenAIResponsesModel(
    model=MODEL,
    openai_client=async_client,
)
agent = Agent(
    name="gpt53-codex-responses-stream-debug",
    model=responses_model,
    instructions="你是一个通用中文助手，不只是代码助手。回答要简洁。",
)

start = time.perf_counter()
streamed = Runner.run_streamed(
    agent,
    input="请用一句中文说明你能做代码以外的事情。",
    max_turns=3,
    run_config=RunConfig(tracing_disabled=True),
)

event_counter = Counter()
raw_counter = Counter()
deltas = []
first_delta_ms = None

async for event in streamed.stream_events():
    et = getattr(event, "type", "unknown")
    event_counter[et] += 1

    if et == "raw_response_event":
        data = getattr(event, "data", None)
        dtype = getattr(data, "type", "unknown")
        raw_counter[dtype] += 1
        if dtype == "response.output_text.delta":
            delta = getattr(data, "delta", "")
            if delta:
                if first_delta_ms is None:
                    first_delta_ms = ms(start)
                deltas.append(delta)
                print(repr(delta), end=" ")

print("\n\nlatency_ms=", ms(start))
print("first_delta_ms=", first_delta_ms)
print("final_output=", streamed.final_output)
print("text_from_deltas=", "".join(deltas))
print("event_counter=", dict(event_counter))
print("raw_counter=", dict(raw_counter))


'当然' '，我' '还' '可以' '帮' '你' '写' '作' '润' '色' '、' '总结' '资料' '、' '制定' '计划' '和' '解' '答' '各' '类' '学习' '生活' '问题' '。' 

latency_ms= 2186
first_delta_ms= 1998
final_output= 当然，我还可以帮你写作润色、总结资料、制定计划和解答各类学习生活问题。
text_from_deltas= 当然，我还可以帮你写作润色、总结资料、制定计划和解答各类学习生活问题。
event_counter= {'agent_updated_stream_event': 1, 'raw_response_event': 33, 'run_item_stream_event': 1}
raw_counter= {'response.created': 1, 'response.in_progress': 1, 'response.output_item.added': 1, 'response.content_part.added': 1, 'response.output_text.delta': 25, 'response.output_text.done': 1, 'response.content_part.done': 1, 'response.output_item.done': 1, 'response.completed': 1}


## 7. 结论记录

本地已初步验证：

- `gpt-5.3-codex` 支持 `/v1/responses`。
- 非流式返回是标准 Responses object，文本在 `output[].content[].text`。
- 流式文本事件是 `response.output_text.delta`，delta 粒度接近 token / 小片段。
- function tool call 在 Responses API 下是 `output[].type == function_call`。
- 该模型不是只能用于 coding，普通中文问答也正常。

如果要接入 ANIFORCE agent adapter，重点适配：

```python
raw_response_event.data.type == "response.output_text.delta"
raw_response_event.data.type == "response.completed"
run_item_stream_event.name == "message_output_created"
# tool call 相关 item 也要看 ResponsesModel 包装后的 RunItem 类型
```
